# SETUP

In [24]:
import os
import pickle
import re
import numpy as np
import pandas as pd

In [25]:
CURRENT_DIR = os.getcwd()

if os.path.isdir(os.path.join(CURRENT_DIR, "data")):
    ROOT_DIR = CURRENT_DIR
elif os.path.isdir(os.path.join(CURRENT_DIR, "..", "data")):
    ROOT_DIR = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
else:
    raise FileNotFoundError(
        "Folder data/ tidak ditemukan. Jalankan notebook dari root project "
        "atau dari folder notebooks/."
    )

DATA_DIR = os.path.join(ROOT_DIR, "data")
MODEL_DIR = os.path.join(ROOT_DIR, "models")
SUBMISSION_DIR = os.path.join(ROOT_DIR, "submissions")

os.makedirs(SUBMISSION_DIR, exist_ok=True)

print("Root project:", ROOT_DIR)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)
print("Submission directory:", SUBMISSION_DIR)

Root project: c:\Downloads\Sem 5\SUP\kagglecomp
Data directory: c:\Downloads\Sem 5\SUP\kagglecomp\data
Model directory: c:\Downloads\Sem 5\SUP\kagglecomp\models
Submission directory: c:\Downloads\Sem 5\SUP\kagglecomp\submissions


# Load Model dan Test

In [26]:
model_path = os.path.join(MODEL_DIR, "best_model.pkl")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")
train_path = os.path.join(DATA_DIR, "train.csv")

for path in [model_path, test_path, sample_path, train_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

with open(model_path, "rb") as file:
    model_bundle = pickle.load(file)

model = model_bundle["model"]
model_name = model_bundle["model_name"]
label_encoder = model_bundle.get("label_encoder")
feature_columns = model_bundle["feature_columns"]
TARGET = model_bundle.get("target_name", "damage_grade")
ID_COL = model_bundle.get("id_column", "building_id")

test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)
train = pd.read_csv(train_path)

print("Model:", model_name)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)
print("Kolom sample:", sample_submission.columns.tolist())
print("\nContoh sample submission:")
display(sample_submission.head())

Model: LightGBM
Train shape: (533465, 17)
Test shape: (228629, 16)
Sample submission shape: (228629, 2)
Kolom sample: ['building_id', 'damage_grade']

Contoh sample submission:


,building_id,damage_grade
0,122207000141,Grade 3
1,284909000871,Grade 1
2,241904000231,Grade 5
3,248308000891,Grade 4
4,300405002981,Grade 2


# Fitur Test

In [27]:
X_test = test.drop(columns=[ID_COL], errors="ignore").copy()

geo_cols = [
    col for col in ["district_id", "vdcmun_id", "ward_id"]
    if col in X_test.columns
]
for col in geo_cols:
    X_test[col] = X_test[col].astype("string")

X_test = X_test.reindex(columns=feature_columns)

print("Jumlah fitur test:", X_test.shape[1])
print("Jumlah baris test:", len(X_test))
display(X_test.head())

Jumlah fitur test: 15
Jumlah baris test: 228629


,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,superstructure
0,12,1205,120504,1,8,240,10,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,has_superstructure_mud_mortar_stone
1,28,2810,281006,3,4,384,22,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
2,24,2409,240909,2,999,384,21,Moderate slope,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
3,24,2412,241202,2,16,527,19,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
4,30,3001,300104,2,6,280,16,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Timber-Planck,Not attached,Rectangular,has_superstructure_mud_mortar_stone


# Prediksi

In [28]:
def predict_labels(model, model_name, X_eval, label_encoder=None):
    pred = np.asarray(model.predict(X_eval))

    if model_name == "XGBoost":
        if label_encoder is None:
            raise ValueError("LabelEncoder tidak ditemukan dalam model bundle.")
        pred = pred.astype(int)
        pred = label_encoder.inverse_transform(pred)

    return pred

predictions = predict_labels(
    model=model,
    model_name=model_name,
    X_eval=X_test,
    label_encoder=label_encoder
)

print("Jumlah prediksi:", len(predictions))
print("Kelas hasil prediksi:", pd.Series(predictions).unique())
display(pd.Series(predictions, name=TARGET).value_counts(dropna=False).sort_index())

Jumlah prediksi: 228629
Kelas hasil prediksi: <StringArray>
['Grade 2', 'Grade 4', 'Grade 3', 'Grade 1', 'Grade 5']
Length: 5, dtype: str


damage_grade
Grade 1    33454
Grade 2    31384
Grade 3    38663
Grade 4    53571
Grade 5    71557
Name: count, dtype: int64

# Submission File

In [29]:
if TARGET not in sample_submission.columns:
    raise ValueError(
        f"Kolom target '{TARGET}' tidak ditemukan pada sample submission. "
        f"Kolom tersedia: {sample_submission.columns.tolist()}"
    )

if len(sample_submission) != len(predictions):
    raise ValueError(
        f"Jumlah prediksi ({len(predictions)}) berbeda dari jumlah baris "
        f"sample submission ({len(sample_submission)})."
    )

submission = sample_submission.copy()

if ID_COL in submission.columns and ID_COL in test.columns:
    if not submission[ID_COL].astype(str).reset_index(drop=True).equals(
        test[ID_COL].astype(str).reset_index(drop=True)
    ):
        raise ValueError(
            "Urutan building_id pada sample_submission.csv berbeda dari test.csv. "
            "Jangan lanjutkan sebelum urutan ID dicocokkan."
        )

submission[TARGET] = predictions

print("Preview sebelum validasi:")
display(submission.head())
print("Dtypes:")
print(submission.dtypes)

Preview sebelum validasi:


,building_id,damage_grade
0,122207000141,Grade 2
1,284909000871,Grade 4
2,241904000231,Grade 3
3,248308000891,Grade 2
4,300405002981,Grade 4


Dtypes:
building_id     int64
damage_grade      str
dtype: object


# Validasi Submission

In [31]:
train_classes = pd.to_numeric(
    train[TARGET]
    .astype(str)
    .str.extract(r"(\d+)", expand=False),
    errors="raise"
).astype(int)
valid_classes = set(train_classes.unique())

submission[TARGET] = pd.to_numeric(
    submission[TARGET]
    .astype(str)
    .str.extract(r"(\d+)", expand=False),
    errors="raise"
).astype(int)
predicted_classes = set(submission[TARGET].unique())

print("Kelas valid dari data training:", sorted(valid_classes))
print("Kelas hasil prediksi:", sorted(predicted_classes))
print("\nDistribusi prediksi:")
print(submission[TARGET].value_counts().sort_index())

assert submission[TARGET].notna().all(), "Ada label prediksi kosong."
assert predicted_classes.issubset(valid_classes), (
    f"Ada kelas prediksi yang tidak ditemukan di training: "
    f"{predicted_classes - valid_classes}"
)

assert list(submission.columns) == list(sample_submission.columns), (
    "Urutan atau nama kolom submission tidak sesuai template."
)
assert len(submission) == len(sample_submission), (
    "Jumlah baris submission tidak sesuai template."
)

Kelas valid dari data training: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Kelas hasil prediksi: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Distribusi prediksi:
damage_grade
1    33454
2    31384
3    38663
4    53571
5    71557
Name: count, dtype: int64


# Simpan ke CSV

In [32]:
submission_path = os.path.join(
    SUBMISSION_DIR,
    "submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

saved_submission = pd.read_csv(submission_path)

print("Baris awal CSV mentah:")
with open(submission_path, "r", encoding="utf-8") as file:
    for _ in range(5):
        print(repr(file.readline().rstrip("\n")))

display(saved_submission.head())

Baris awal CSV mentah:
'building_id,damage_grade'
'122207000141,2'
'284909000871,4'
'241904000231,3'
'248308000891,2'


,building_id,damage_grade
0,122207000141,2
1,284909000871,4
2,241904000231,3
3,248308000891,2
4,300405002981,4
